# NB-02_controles_y_ajustes_iniciales

Process Flow SAS: **Controles y ajustes iniciales** — `PFD-I32F27oZ8ICuby6y`

In [ ]:
# ========= Parámetros =========
# Variables macro del SAS original. El .egp NO las define (venían del
# entorno SAS): su valor sale de la entrevista B4 o de
# project_config.yaml → run.macro_params, o se inyecta acá
# (celda 'parameters' de papermill).

ANIO = None  # &ANIO — nadie declaró su valor
TRIM = None  # &TRIM — nadie declaró su valor
anio = None  # &anio — nadie declaró su valor

faltantes = [n for n, v in {"ANIO": ANIO, "TRIM": TRIM, "anio": anio}.items() if v is None]
if faltantes:
    raise ValueError(f"Parámetros sin valor: {faltantes}")

In [ ]:
# ========= Celda 1: Configuración =========
import pandas as pd
import numpy as np
import os
import sqlalchemy
import datetime
from pathlib import Path
import bcchapi
from sqlalchemy import text
import pyreadstat

# Conexión a BD — editable acá; SASMIG_DB_URL (orquestador) tiene
# prioridad si está definida (SUPUESTO: verificar servidor y base
# antes de correr contra datos reales).
config_db = (
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=PLATDAT,1433;"
    "DATABASE=GOBGENER;"
    "Authentication=ActiveDirectoryIntegrated;"
    "Encrypt=yes;"
    "TrustServerCertificate=no;"
    "MARS_Connection=Yes;"
)
engine = sqlalchemy.create_engine(
    os.environ.get("SASMIG_DB_URL", f"mssql+pyodbc:///?odbc_connect={config_db}"),
    pool_pre_ping=True,
    fast_executemany=True,
)
# Sesión de BD del notebook — espejo de la sesión WORK de SAS: las
# tablas temporales #tmp viven en ESTA conexión y mueren al cerrar el
# kernel. AUTOCOMMIT: cada statement commitea, como los pasos de SAS.
work_conn = engine.connect().execution_options(isolation_level="AUTOCOMMIT")

# Logging liviano de resultados — aprobado en la entrevista (Fase 4)
_LOG_PATH = Path("log") / "NB-02_controles_y_ajustes_iniciales.log"
_LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
def _log(label, value=None):
    """Una línea por celda: imprime y persiste. Jamás rompe la corrida."""
    try:
        if hasattr(value, "shape"):
            detail = f"{value.shape[0]} filas x {value.shape[1]} cols"
        elif isinstance(value, int):
            detail = f"{value} filas"
        elif value is None:
            detail = "ok"
        else:
            detail = str(value)
        line = f"[{datetime.datetime.now():%Y-%m-%d %H:%M:%S}] {label}: {detail}"
        print(line)
        with open(_LOG_PATH, "a", encoding="utf-8") as fh:
            fh.write(line + "\n")
    except Exception:
        pass  # el log nunca puede tumbar el notebook
with open(_LOG_PATH, "a", encoding="utf-8") as _fh:
    _fh.write(f"\n=== corrida {datetime.datetime.now():%Y-%m-%d %H:%M:%S} ===\n")


## S1

Arma las bases auxiliares de síntesis: SIFMI por sector y entrada, promedio trimestral de dividendos de hogares, agregados de cuentas nacionales trimestrales tomados de la API del Banco Central, utilidades reinvertidas del sector financiero y los ajustes varios del cierre / Reclasifica los bonos de renta fija externa como préstamos de largo plazo y traslada el activo AF.32 del resto del mundo con contraparte 36 hacia empresas, incorporando ambos ajustes a la tabla de ajustes varios

*confianza: medium · verificador: approve · SAS: PROC IMPORT XLSX + PROC SQL con UNION ALL / GROUP BY + PROC HTTP a la API BDE + PROC APPEND y DELETE sobre tablas de la BD + PROC SQL UPDATE + CREATE TABLE con GROUP BY + DATA step SET (concatenación) sobre TABLAS.AJ_VARIOS*

In [ ]:
# ========= S1 =========
# /******************SIFMI TRIMESTRAL-BENCH********************/
# /*IMPORTA DATA*/
ruta_sifmi = Path("data") / "CONTROLES" / "SIFMI.xlsx"
sifmi = pd.read_excel(ruta_sifmi, sheet_name="SIFMI_SAS")
_log("sifmi", sifmi)


In [ ]:
# /*CREA BASE DE DATOS SIFMI. CIERRE 2021: INCORPORA SECTORES GOB, SEGUROS Y AUXILIARES*/
# 12 bloques UNION ALL sobre WORK.SIFMI: 6 de pagados (signo invertido) y 6 de recibidos
fecha_hoy = pd.Timestamp.today().normalize()

_bloques_sifmi = [
    # (SECTOR, C_CAGENTE, C_ENTRADA, serie de DATO)
    (51, "321", "D", sifmi["Empresas_pagados"] * -1),
    (511, "321", "D", sifmi["Hogares_pagados"] * -1),
    (41, "321", "D", sifmi["Gob_pagados"] * -1),
    (35, "321", "D", sifmi["Seg_pagados"] * -1),
    (36, "321", "D", sifmi["Aux_pagados"] * -1),
    (321, "53", "H", (sifmi["Hogares_pagados"] + sifmi["Empresas_pagados"] + sifmi["Gob_pagados"] + sifmi["Seg_pagados"] + sifmi["Aux_pagados"]) * -1),
    (51, "321", "H", sifmi["Empresas_recibidos"]),
    (511, "321", "H", sifmi["Hogares_recibidos"]),
    (41, "321", "H", sifmi["Gob_recibidos"]),
    (35, "321", "H", sifmi["Seg_recibidos"]),
    (36, "321", "H", sifmi["Aux_recibidos"]),
    (321, "53", "D", sifmi["Hogares_recibidos"] + sifmi["Empresas_recibidos"] + sifmi["Gob_recibidos"] + sifmi["Seg_recibidos"] + sifmi["Aux_recibidos"]),
]

_partes_sifmi = []
for _sector, _cagente, _entrada, _dato in _bloques_sifmi:
    _partes_sifmi.append(pd.DataFrame({
        "MONEDA": "P",
        "AÑO": sifmi["Año"],
        "TRIM": sifmi["Trimestre"],
        "SECTOR": _sector,
        "C_CUENTA": "YG",
        "C_CAGENTE": _cagente,
        "C_ENTRADA": _entrada,
        "DATO": _dato.values,
        "C_SCN": "D.41",
        "N_SCN": "Intereses",
        "FUENTE": "DI_Aj_SIFMI",
        "FECHA": fecha_hoy,
    }))

tablas_sifmi = pd.concat(_partes_sifmi, ignore_index=True)
# PROC SQL; DELETE FROM TABLAS.SIFMI WHERE AÑO=.;
tablas_sifmi = tablas_sifmi[tablas_sifmi["AÑO"].notna()].reset_index(drop=True)
_log("tablas_sifmi", tablas_sifmi)


In [ ]:
# CREATE TABLE TABLAS.SIFMI: el SAS reemplazaba la tabla → DELETE sin WHERE + append
with engine.begin() as conn:
    res = conn.execute(text("DELETE FROM TABLAS.dbo.SIFMI"))
    _log("DELETE TABLAS.dbo.SIFMI", res.rowcount)
tablas_sifmi.to_sql("SIFMI", engine, schema="dbo", if_exists="append", index=False)


In [ ]:
# /******************DIVIDENDOS HOGARES********************/
# /*CALCULA PROMEDIO DEL TRIMESTRE A TRABAJAR. AJUSTE DE INICIO DEL PERIODO EN EL PROCESO DE SÍNTESIS PARA EL PERIODO DE COYUNTURA*/
sql_rp_hh_sum = """
SELECT T1.MONEDA, T1.AÑO, T1.TRIM, T1.SECTOR, T1.C_CUENTA, T1.C_CAGENTE,
       T1.C_ENTRADA, T1.C_SCN, T1.N_SCN, T1.FUENTE, SUM(T1.DATO) AS DATO
FROM TABLAS.dbo.RP_HH T1
WHERE T1.AÑO >= 2008 AND T1.TRIM = :trim
GROUP BY T1.MONEDA, T1.AÑO, T1.TRIM, T1.SECTOR, T1.C_CUENTA, T1.C_CAGENTE,
         T1.C_ENTRADA, T1.C_SCN, T1.N_SCN, T1.FUENTE
"""
rp_hh_sum = pd.read_sql(text(sql_rp_hh_sum), engine, params={"trim": TRIM})

# /*PARA ELIMINAR PERIODO DE COYUNTURA EN CASO QUE SE CORRA ESTE PROG VARIAS VECES*/
rp_hh_sum = rp_hh_sum[~((rp_hh_sum["AÑO"] == ANIO) & (rp_hh_sum["TRIM"] == TRIM))].reset_index(drop=True)
_log("rp_hh_sum", rp_hh_sum)


In [ ]:
# Promedio de los trimestres históricos, imputado al año de coyuntura (&ANIO)
_claves_rp = ["MONEDA", "TRIM", "SECTOR", "C_CUENTA", "C_CAGENTE", "C_ENTRADA", "C_SCN", "N_SCN", "FUENTE"]
rp_hh_av = (
    rp_hh_sum.groupby(_claves_rp, dropna=False, as_index=False)["DATO"].mean()
)
rp_hh_av["AÑO"] = ANIO
rp_hh_av["FECHA"] = pd.Timestamp.today().normalize()
rp_hh_av["PROC"] = "P"
rp_hh_av = rp_hh_av[["MONEDA", "AÑO", "TRIM", "SECTOR", "C_CUENTA", "C_CAGENTE",
                    "C_ENTRADA", "C_SCN", "N_SCN", "FUENTE", "DATO", "FECHA", "PROC"]]
_log("rp_hh_av", rp_hh_av)


In [ ]:
# /*ELIMINA DATOS DE COYUNTURA EN TABLA PRINCIPAL*/
# El DELETE correlacionado usa las combinaciones AÑO/TRIM/PROC presentes en RP_HH_AV
_combos_rp = rp_hh_av[["AÑO", "TRIM", "PROC"]].drop_duplicates().to_dict("records")
sql_del_rp_hh = """
DELETE FROM TABLAS.dbo.RP_HH
WHERE AÑO = :anio AND TRIM = :trim AND PROC = :proc
"""
with engine.begin() as conn:
    _borradas_rp = 0
    for _c in _combos_rp:
        res = conn.execute(text(sql_del_rp_hh), {"anio": int(_c["AÑO"]), "trim": int(_c["TRIM"]), "proc": _c["PROC"]})
        _borradas_rp += res.rowcount
_log("DELETE TABLAS.dbo.RP_HH", _borradas_rp)


In [ ]:
# /*ANEXA PROMEDIO DIVIDENDOS A BASE RP_HH*/
# PROC APPEND ... FORCE: acumula, re-ejecutar duplica igual que el SAS original
rp_hh_av.to_sql("RP_HH", engine, schema="dbo", if_exists="append", index=False)
_log("APPEND TABLAS.dbo.RP_HH", len(rp_hh_av))


In [ ]:
# /******************BASE CON VARIABLES DE LAS CNT A UTILIZAR PARA AJUSTAR DATOS DE LAS CNSI********************/
# /*IMPORTA DATA DESDE ARCHIVO GENERADO POR API*/
# /*obtiene datos directo desde API web*/
# Cliente de la API BDE del Banco Central (host declarado con mode=sdk → bcchapi)
siete = bcchapi.Siete(os.environ["BDE_USER"], os.environ["BDE_PASS"])


def _serie_bde(codigo_serie):
    """Descarga una serie de la API BDE y devuelve AÑO, TRIM (mes del indexDateString) y VALOR."""
    _df = siete.cuadro(series=[codigo_serie], nombres=["value"]).reset_index()
    _df = _df.rename(columns={_df.columns[0]: "fecha"})
    _df["fecha"] = pd.to_datetime(_df["fecha"])
    resp = pd.DataFrame({
        "AÑO": _df["fecha"].dt.year.astype("Int64"),
        "TRIM": _df["fecha"].dt.month.astype("Int64"),
        "VALOR": pd.to_numeric(_df["value"], errors="coerce"),
    })
    return resp[resp["AÑO"].notna()].reset_index(drop=True)


In [ ]:
# /*PIB a precios corrientes*/
resp = _serie_bde("F032.PIB.FLU.N.CLP.EP18.Z.Z.0.T")
pib = pd.DataFrame({
    "AÑO": resp["AÑO"],
    "TRIM": resp["TRIM"],
    "DATO": resp["VALOR"] * 1000,
    "FECHA": pd.Timestamp.today().normalize(),
})
# UPDATE tablas.PIB SET TRIM=2 WHERE TRIM=4 / 3 WHERE 7 / 4 WHERE 10
pib["TRIM"] = pib["TRIM"].replace({4: 2, 7: 3, 10: 4})
_log("pib", pib)


In [ ]:
# CREATE TABLE tablas.PIB: el SAS reemplazaba la tabla → DELETE sin WHERE + append
with engine.begin() as conn:
    res = conn.execute(text("DELETE FROM TABLAS.dbo.PIB"))
    _log("DELETE TABLAS.dbo.PIB", res.rowcount)
pib.to_sql("PIB", engine, schema="dbo", if_exists="append", index=False)


In [ ]:
# Series de las CNT: mismo layout para las 9 (solo cambian sector, cuenta, entrada, C_SCN y nombre)
def _serie_cnt(codigo_serie, sector, c_cuenta, c_entrada, c_scn, n_scn):
    _obs = _serie_bde(codigo_serie)
    _obs = _obs[_obs["AÑO"] >= 2003]
    return pd.DataFrame({
        "AÑO": _obs["AÑO"].values,
        "TRIM": _obs["TRIM"].values,
        "SECTOR": sector,
        "C_CUENTA": c_cuenta,
        "C_ENTRADA": c_entrada,
        "DATO": (_obs["VALOR"] * 1000).values,
        "C_SCN": c_scn,
        "N_SCN": n_scn,
        "FUENTE": "CNT",
        "FECHA": pd.Timestamp.today().normalize(),
    })


# /*Ingreso de los factores recibidos*/
serie_1 = _serie_cnt("F033.IRM.FLU.N.CLP.EP18.0.T", 6, "YG", "D", "D.4", "Ingreso de factores recibidos del RM")
# /*Ingreso de los factores pagados al RM*/
serie_2 = _serie_cnt("F033.IRMP.FLU.N.CLP.EP18.0.T", 6, "YG", "H", "D.4", "Ingreso de factores pagados al RM")
# /*Transferencias corrientes recibidas del exterior*/
serie_3 = _serie_cnt("F033.TCE.FLU.N.CLP.EP18.0.T", 6, "YG", "D", "D.7", "Transferencias corrientes recibidos del RM")
# /*Transferencias corrientes pagadas al exterior*/
serie_4 = _serie_cnt("F033.TCEP.FLU.N.CLP.EP18.0.T", 6, "YG", "H", "D.7", "Transferencias corrientes pagados al RM")
# /*Ahorro externo*/
serie_5 = _serie_cnt("F033.AEX.FLU.N.CLP.EP18.0.T", 6, "YG", "D", "B.8", "Ahorro externo")
# /*Formacion_bruta_capital fijo*/
serie_6 = _serie_cnt("F033.FKF.FLU.N.CLP.EP18.0.T", 53, "Capital", "D", "P.51", "Formación bruta de capital fijo")
# /*Variacion_Existencias*/
serie_7 = _serie_cnt("F033.VAX.FLU.N.CLP.EP18.0.T", 53, "Capital", "D", "P.52", "Variación de existencias")
# /*Exportaciones*/
serie_8 = _serie_cnt("F033.XBS.FLU.N.CLP.EP18.0.T", 6, "Producción", "D", "P.7", "Importación de bienes y servicios")
# /*Importaciones*/
serie_9 = _serie_cnt("F033.IBS.FLU.N.CLP.EP18.0.T", 6, "Producción", "H", "P.6", "Exportación de bienes y servicios")
_log("serie_9", serie_9)


In [ ]:
# /*une base de datos CNT*/
cnt = pd.concat([serie_1, serie_2, serie_3, serie_4, serie_5, serie_6, serie_7, serie_8, serie_9], ignore_index=True)
# UPDATE TABLAS.CNT SET TRIM=2 WHERE TRIM=4 / 3 WHERE 7 / 4 WHERE 10
cnt["TRIM"] = cnt["TRIM"].replace({4: 2, 7: 3, 10: 4})
_log("cnt", cnt)


In [ ]:
# DATA tablas.CNT: el SAS reemplazaba la tabla → DELETE sin WHERE + append
with engine.begin() as conn:
    res = conn.execute(text("DELETE FROM TABLAS.dbo.CNT"))
    _log("DELETE TABLAS.dbo.CNT", res.rowcount)
cnt.to_sql("CNT", engine, schema="dbo", if_exists="append", index=False)


In [ ]:
# proc sql; drop table serie_1 ... serie_9  → se liberan los DataFrames intermedios
for _s in ["serie_1", "serie_2", "serie_3", "serie_4", "serie_5", "serie_6", "serie_7", "serie_8", "serie_9"]:
    del globals()[_s]


In [ ]:
# /******************UTILIDADES REINVERTIDAS DEL SECTOR FINANCIERO********************/
# /*IMPORTA DATA FINAL DE UTILIDADES REINVERTIDAS PAGADAS POR EL SECTOR, NUEVO CALCULO TRIMESTRAL CR18. ADEMÁS INCORPORA APERTURA ENTRE BANCOS Y SEGUROS*/
ruta_ur_sf = Path("data") / "INFO_AUX" / "UT_REINVERTIDAS_CR18.xlsx"
ur_sf_cr18 = pd.read_excel(ruta_ur_sf, sheet_name="UR_SF", skiprows=1)
# /*NO SE INCORPORA APERTURA PORQUE AFECTA MUCHO EL PTMO NETO DE LOS SEGUROS*/
ur_sf_cr18["SECTOR"] = ur_sf_cr18["SECTOR"].replace({35: 321})
_log("ur_sf_cr18", ur_sf_cr18)


In [ ]:
# PROC IMPORT out=TABLAS.UR_SF_CR18 (replace) → DELETE sin WHERE + append
with engine.begin() as conn:
    res = conn.execute(text("DELETE FROM TABLAS.dbo.UR_SF_CR18"))
    _log("DELETE TABLAS.dbo.UR_SF_CR18", res.rowcount)
ur_sf_cr18.to_sql("UR_SF_CR18", engine, schema="dbo", if_exists="append", index=False)


In [ ]:
# /*IMPORTA DATA DE CCAS*/
ruta_t_ccast = Path("data") / "INFO_AUX" / "T_CCAST.xlsx"
t_ccast = pd.read_excel(ruta_t_ccast, sheet_name="T_CCAST")
_log("t_ccast", t_ccast)


In [ ]:
# DATA tablas.T_CCAST; SET WORK.T_CCAST → reemplazo de la tabla (DELETE sin WHERE + append)
with engine.begin() as conn:
    res = conn.execute(text("DELETE FROM TABLAS.dbo.T_CCAST"))
    _log("DELETE TABLAS.dbo.T_CCAST", res.rowcount)
t_ccast.to_sql("T_CCAST", engine, schema="dbo", if_exists="append", index=False)


In [ ]:
# /*ELIMINA DE AJUSTE BONOS AÑO DE COYUNTURA PARA RECALCULAR DENUEVO POR CAMBIO DE CUENTAS INDIVIDUALES*/
with engine.begin() as conn:
    res = conn.execute(text("DELETE FROM TABLAS.dbo.AJUSTE_BONOS WHERE AÑO >= :anio"), {"anio": ANIO})
    _log("DELETE TABLAS.dbo.AJUSTE_BONOS", res.rowcount)


In [ ]:
# /*IMPORTA AJUSTES VARIOS DEP Y ACCIONES*/
ruta_aj_cnsi = Path("data") / "INFO_AUX" / "aj_cnsi.xlsx"
aj_varios = pd.read_excel(ruta_aj_cnsi, sheet_name="AJUSTES_VARIOS")
_log("aj_varios", aj_varios)


In [ ]:
# /*CIERRE 2021: IMPORTA AJUSTES TRANSFERENCIAS CORREINTES DE EMPRESAS POR CDR18*/
aj_d443_cr18 = pd.read_excel(ruta_aj_cnsi, sheet_name="base_aj_d443_cr18")
# proc sql; delete from AJ_d443_CR18 where año=.;
_col_anio_d443 = "año" if "año" in aj_d443_cr18.columns else "AÑO"
aj_d443_cr18 = aj_d443_cr18[aj_d443_cr18[_col_anio_d443].notna()].reset_index(drop=True)
_log("aj_d443_cr18", aj_d443_cr18)


In [ ]:
# DATA TABLAS.AJ_VARIOS; SET TABLAS.AJ_VARIOS AJ_d443_CR18;
aj_varios = pd.concat([aj_varios, aj_d443_cr18], ignore_index=True)
# PROC SQL; DROP TABLE AJ_d443_CR18;
del aj_d443_cr18
_log("aj_varios", aj_varios)


In [ ]:
# PROC IMPORT out=TABLAS.AJ_VARIOS (replace) + concatenación → DELETE sin WHERE + append
with engine.begin() as conn:
    res = conn.execute(text("DELETE FROM TABLAS.dbo.AJ_VARIOS"))
    _log("DELETE TABLAS.dbo.AJ_VARIOS", res.rowcount)
aj_varios.to_sql("AJ_VARIOS", engine, schema="dbo", if_exists="append", index=False)


In [ ]:
# /*CIERRE 2021: INCORPORA AJUSTE A FBCF SECTOR FINANCIERO*/
ruta_fbcf_sf = Path("data") / "INFO_AUX" / "aj_fbcf_sf.xlsx"
fbcf_sf = pd.read_excel(ruta_fbcf_sf, sheet_name="BASE")
_log("fbcf_sf", fbcf_sf)


In [ ]:
# PROC IMPORT out=tablas.FBCF_SF (replace) → DELETE sin WHERE + append
with engine.begin() as conn:
    res = conn.execute(text("DELETE FROM TABLAS.dbo.FBCF_SF"))
    _log("DELETE TABLAS.dbo.FBCF_SF", res.rowcount)
fbcf_sf.to_sql("FBCF_SF", engine, schema="dbo", if_exists="append", index=False)


In [ ]:
# /******************LIMPIA TABLAS TEMPORALES DEL FLUJO DE PROCESO********************/
# DROP TABLE SIFMI, RP_HH_SUM, RP_HH_AV, T_CCAST, ... (solo se liberan las que este tramo creó)
for _t in ["sifmi", "rp_hh_sum", "rp_hh_av", "t_ccast"]:
    globals().pop(_t, None)


In [ ]:
# /*SE INCORPORA EN CIERRE 2022Q2. DATA BONOS EMITIDOS EN EL EXTERIOR POR NO REGULADOS DEL SECTOR FINANCIERO SECTOR=36912.
# EN CIERRE DE AÑO IMPUTAR TODA LA SERIE. CIERRE 2022: SE INCORPORA AJUSTE PARA TODA LA SERIE PARA SER CONSISTENTES
# CIERRE 2025Q2: INCORPORA EL EMISOR 33*/
raise NotImplementedError(
    "BONOS_RF_EXT / BONOS_RF_EXT_BI / RP_AUXFIN: la fuente es el dataset SAS "
    "'/sasdata/BCCH/GEM_DCNI/02_CNSI/05_DCV/Data/HSS/BASE_DEUDA_EMV.sas7bdat', "
    "que no está declarado como tabla de BD ni como archivo disponible en el workspace. "
    "Falta definir su origen (ruta de datos o tabla equivalente) para leerlo con pyreadstat."
)


In [ ]:
# PROC SQL UPDATE sobre WORK.BONOS_RF_EXT: se reclasifica el instrumento a préstamos
# de largo plazo del sector 321 y se invierte el signo del dato.
bonos_rf_ext["C_SCN"] = "AF.42"
bonos_rf_ext["N_SCN"] = "Préstamos a largo plazo"
bonos_rf_ext["C_CAGENTE"] = "321"
bonos_rf_ext["dato"] = bonos_rf_ext["dato"] * -1
_log("bonos_rf_ext", bonos_rf_ext)


In [ ]:
# DATA TABLAS.AJ_VARIOS; SET TABLAS.AJ_VARIOS BONOS_RF_EXT; -> reemplazo de la tabla
# con el contenido actual más los bonos de renta fija externa reclasificados.
aj_varios = pd.read_sql(text("SELECT * FROM TABLAS.dbo.AJ_VARIOS"), engine)
aj_varios = pd.concat([aj_varios, bonos_rf_ext], ignore_index=True)
_log("aj_varios", aj_varios)


In [ ]:
with engine.begin() as conn:
    res = conn.execute(text("DELETE FROM TABLAS.dbo.AJ_VARIOS"))
    _log("DELETE TABLAS.dbo.AJ_VARIOS", res.rowcount)
aj_varios.to_sql("AJ_VARIOS", engine, schema="dbo", if_exists="append", index=False)


In [ ]:
# ELIMINA BONO DEL RM ACTIVO CON CA 36 DESDE 2022, PARA CONCILIAR BIEN CON LO IMPUTADO EN LA CI DE SECTOR 36
# LO QUE ESTA INICIALMENTE SE LLEVA A EMPRESAS
# La fuente es el dataset SAS BD_CTSI_CIERRE.sas7bdat (ruta del servidor SAS,
# reubicada bajo data/ según la convención del proyecto).
ruta_bd_ctsi_cierre = Path("data") / "02_PRE_SINTESIS" / "BD_CTSI_CIERRE.sas7bdat"
bd_ctsi_cierre, _meta_bd_ctsi_cierre = pyreadstat.read_sas7bdat(str(ruta_bd_ctsi_cierre))
_log("bd_ctsi_cierre", bd_ctsi_cierre)


In [ ]:
_filtro_af32 = (
    (bd_ctsi_cierre["C_ENTRADA"] == "D")
    & (bd_ctsi_cierre["C_SCN"] == "AF.32")
    & (bd_ctsi_cierre["SECTOR"] == 6)
    & (bd_ctsi_cierre["C_CAGENTE"] == "36")
    & (bd_ctsi_cierre["FUENTE"] == "CI")
)
_base_af32 = bd_ctsi_cierre[_filtro_af32].copy()
_claves_af32 = ["MONEDA", "A\u00d1O", "TRIM", "C_CUENTA", "C_ENTRADA", "C_SCN", "N_SCN"]

# LO QUE ESTA INICIALMENTE SE LLEVA A EMPRESAS
af32_51_6 = (
    _base_af32.groupby(_claves_af32, as_index=False, dropna=False)
    .agg(DATO=("DATO", "sum"), SECTOR=("SECTOR", "first"), FUENTE=("FUENTE", "first"))
)
af32_51_6["C_CAGENTE"] = "51021"
af32_51_6 = af32_51_6[
    ["MONEDA", "A\u00d1O", "TRIM", "SECTOR", "C_CAGENTE", "C_CUENTA", "C_ENTRADA", "DATO", "C_SCN", "N_SCN", "FUENTE"]
]
_log("af32_51_6", af32_51_6)


In [ ]:
# LO QUE ESTA INICIALMENTE SE RESTA, EXCEPTO BI DE TRIM=1
af32_36_6 = (
    _base_af32.groupby(_claves_af32, as_index=False, dropna=False)
    .agg(
        DATO=("DATO", "sum"),
        SECTOR=("SECTOR", "first"),
        C_CAGENTE=("C_CAGENTE", "first"),
        FUENTE=("FUENTE", "first"),
    )
)
af32_36_6["DATO"] = af32_36_6["DATO"] * -1
af32_36_6 = af32_36_6[
    ["MONEDA", "A\u00d1O", "TRIM", "SECTOR", "C_CAGENTE", "C_CUENTA", "C_ENTRADA", "DATO", "C_SCN", "N_SCN", "FUENTE"]
]
_log("af32_36_6", af32_36_6)


In [ ]:
# DATA TABLAS.AJ_VARIOS; SET TABLAS.AJ_VARIOS AF32_51_6 AF32_36_6; -> reemplazo de la tabla
aj_varios = pd.concat([aj_varios, af32_51_6, af32_36_6], ignore_index=True)
_log("aj_varios", aj_varios)


In [ ]:
with engine.begin() as conn:
    res = conn.execute(text("DELETE FROM TABLAS.dbo.AJ_VARIOS"))
    _log("DELETE TABLAS.dbo.AJ_VARIOS", res.rowcount)
aj_varios.to_sql("AJ_VARIOS", engine, schema="dbo", if_exists="append", index=False)


In [ ]:
# PROC SQL; DROP TABLE AF32_51_6, AF32_36_6, AF32_36_6_VOL; -> los WORK viven en memoria
for _t in ["af32_51_6", "af32_36_6"]:
    globals().pop(_t, None)


## S1_d

Calcula por año y trimestre las transferencias de capital del gobierno a empresas públicas y reemplaza el periodo de coyuntura en la tabla principal de transferencias

*confianza: medium · verificador: approve · SAS: PROC SQL CREATE TABLE con LEFT JOIN y agregación + DELETE + PROC DATASETS APPEND*

In [ ]:
# ========= S1_d =========
# /*obtiene transferencias de capital a empresas para imputar en la síntesis*/
# /*año debe ser mayor o igual a 2005 en cierre de año y el corriente en coyuntura*/
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_cgr"))
# año interpolado como entero (no :param) para que la #tmp sobreviva
sql_ejec_cgr = f"""
SELECT *
INTO #ejec_cgr
FROM GOBGENER.dbo.EJECUCION
WHERE AÑO >= {int(ANIO)}
"""
res = work_conn.execute(text(sql_ejec_cgr))
_log("#ejec_cgr", res.rowcount)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #tk_ejec"))
# El trimestre se deriva del mes (1-3=1, 4-6=2, 7-9=3, resto=4) y se agrupa por año y trimestre
sql_tk_ejec = """
SELECT t1.AÑO,
       (CASE WHEN t1.MES IN (1,2,3) THEN 1
             WHEN t1.MES IN (4,5,6) THEN 2
             WHEN t1.MES IN (7,8,9) THEN 3
             ELSE 4 END) AS TRIM,
       5101 AS C_SI,
       'D.9' AS C_INSTRUMENTO_SCN,
       'Capital' AS C_CUENTA,
       'H' AS C_ENTRADA,
       SUM(t1.DEVENG/1000000.0) AS Dato
INTO #tk_ejec
FROM #ejec_cgr t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_2 t2
       ON  t1.C_PARTIDA     = t2.C_PARTIDA
       AND t1.C_CAPITULO    = t2.C_CAPITULO
       AND t1.C_PROGRAMA    = t2.C_PROGRAMA
       AND t1.C_ENTIDAD     = t2.ENTIDAD
       AND t1.C_TIPO_CUENTA = t2.T_CUENTA
       AND t1.C_CUENTA      = t2.C_CUENTA
       AND t1.C_ITEM        = t2.C_ITEM
       AND t1.C_ASIGNACION  = t2.C_ASIGNACION
       AND t1.C_ANALITICO   = t2.C_ANALITICO
WHERE t1.C_CUENTA IN ('05','13','24','33')
  AND (t2.OBS IS NULL OR t2.OBS NOT IN ('CR18_difcoy','CR18_difact'))
  AND t1.C_ENTIDAD NOT IN (5601,10201)
  AND (t2.C_SCN IS NULL OR t2.C_SCN <> 'TC')
  AND t1.MONEDA = 'P'
  AND t2.C_SCN IN ('D91','D92','D93','D99')
  AND t1.C_TIPO_CUENTA = 'G'
  AND t2.C_CAGENTE IN ('S11','S11_EPU','tkemppúb')
  AND t2.N_CUENTA = 'capital'
GROUP BY t1.AÑO,
         (CASE WHEN t1.MES IN (1,2,3) THEN 1
               WHEN t1.MES IN (4,5,6) THEN 2
               WHEN t1.MES IN (7,8,9) THEN 3
               ELSE 4 END)
"""
res = work_conn.execute(text(sql_tk_ejec))
_log("#tk_ejec", res.rowcount)


In [ ]:
# /*ELIMINA DATOS DE AÑO DE COYUNTURA EN TABLA PRINCIPAL*/
with engine.begin() as conn:
    res = conn.execute(text("DELETE FROM TABLAS.dbo.T_TK_GG_EPU WHERE AÑO >= :anio"), {"anio": int(ANIO)})
    _log("DELETE TABLAS.dbo.T_TK_GG_EPU", res.rowcount)


In [ ]:
res = work_conn.execute(text("DELETE FROM #tk_ejec WHERE AÑO < :anio"), {"anio": int(ANIO)})
_log("DELETE #tk_ejec", res.rowcount)


In [ ]:
# /*ANEXA TK DE COYUNTURA A TABLA PRINCIPAL*/
# INSERT server-side: el append no baja los datos a pandas (FORCE alinea por nombre)
cols_tk_gg_epu = "AÑO, TRIM, C_SI, C_INSTRUMENTO_SCN, C_CUENTA, C_ENTRADA, Dato"
sql_append_tk_gg_epu = f"""
INSERT INTO TABLAS.dbo.T_TK_GG_EPU ({cols_tk_gg_epu})
SELECT {cols_tk_gg_epu}
FROM #tk_ejec
"""
res = work_conn.execute(text(sql_append_tk_gg_epu))
_log("APPEND TABLAS.dbo.T_TK_GG_EPU", res.rowcount)


In [ ]:
for t in ["#ejec_cgr", "#tk_ejec"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


## S1_b

Calcula el porcentaje de los depósitos de fondos mutuos que corresponde a hogares, por año, trimestre y sector, completa los años 2003-2005 con el valor de 2006T1 y actualiza la serie desde el año de proceso

*confianza: low · verificador: approve · SAS: PROC IMPORT XLSX + múltiples PROC SQL (agregaciones, joins, UPDATE) + DATA step de concatenación + PROC DATASETS APPEND*

In [ ]:
# ========= S1_b =========
# CALCULA % DE DEPÓSITOS PARA HOGARES EN FFMM
# El SAS lee IDENTIFICA_FFMM.sas7bdat directamente desde el servidor SAS
raise NotImplementedError(
    "WORK.ID_FM: el SAS consulta el dataset SAS /sasdata/BCCH/GEM_DCNI/02_CNSI/12_SI_FI/33901_FFMM/IDENTIFICA_FFMM.sas7bdat, "
    "cuyo origen no está declarado como tabla de BD ni como archivo de entrada del proyecto. "
    "Falta definir de dónde se obtiene run_fondo/tipo_ffmm (tabla en la base o archivo en data/33901_FFMM/)."
)


In [ ]:
# 1. CALCULA % A HOGARES POR ROL DEL FONDO
ruta_patrimonio = Path("data") / "mensual" / "BD_Patrimonio.xlsx"
base_datos = pd.read_excel(ruta_patrimonio, sheet_name="base_datos", header=0)
_log("base_datos", base_datos)


In [ ]:
base_datos.loc[(base_datos["AÑO"] < 2017) & (base_datos["MES"].isna()), "MES"] = 12
# la base de patrimonio se sube como #tmp de sesión para operar en SQL
base_datos.to_sql("#base_datos", work_conn, if_exists="replace", index=False)
_log("base_datos", base_datos)


In [ ]:
# DATO A HOGARES
work_conn.execute(text("DROP TABLE IF EXISTS #dato_hh"))
sql_dato_hh = """
SELECT AÑO, MES, RUN, SUBSTRING(RUN, 1, 4) AS RUN_SDV,
       SUM(CASE WHEN DESTINO = 'EMPRE_HOGAR' THEN PATRI_T / 2 ELSE PATRI_T END) AS DATO,
       MIN(tipo_fondo) AS tipo_fondo
INTO #dato_hh
FROM #base_datos
WHERE DESTINO IN ('HOGARES', 'EMPRE_HOGAR')
GROUP BY AÑO, MES, RUN, tipo_fondo
"""
res = work_conn.execute(text(sql_dato_hh))
_log("#dato_hh", res.rowcount)


In [ ]:
# DATO TOTAL
work_conn.execute(text("DROP TABLE IF EXISTS #dato_tot"))
sql_dato_tot = """
SELECT AÑO, MES, RUN, MIN(SUBSTRING(RUN, 1, 4)) AS RUN_SDV, SUM(PATRI_T) AS DATO
INTO #dato_tot
FROM #base_datos
GROUP BY AÑO, MES, RUN
"""
res = work_conn.execute(text(sql_dato_tot))
_log("#dato_tot", res.rowcount)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #dato_est"))
sql_dato_est = """
SELECT T1.AÑO, T1.MES, T1.RUN, TRY_CAST(T1.RUN_SDV AS float) AS RUN_SDV,
       T1.tipo_fondo, T1.DATO / T2.DATO AS EST
INTO #dato_est
FROM #dato_hh T1, #dato_tot T2
WHERE T1.AÑO = T2.AÑO AND T1.MES = T2.MES AND T1.RUN = T2.RUN
"""
res = work_conn.execute(text(sql_dato_est))
dato_est = pd.read_sql(text("SELECT * FROM #dato_est"), work_conn)
_log("dato_est", dato_est)


In [ ]:
for t in ["#base_datos", "#dato_hh", "#dato_tot"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# 2. OBTIENE DATOS A APLICAR PORCENTAJE
# El SAS lee las carteras de inversión directamente desde datasets SAS del servidor
raise NotImplementedError(
    "WORK.DATA_FM_NAC / WORK.DATA_FM_EXT: el SAS lee CARTERA_INV_NACIONAL.sas7bdat y CARTERA_INV_EXTERNA.sas7bdat "
    "desde /sasdata/BCCH/GEM_DCNI/02_CNSI/12_SI_FI/33901_FFMM/, orígenes no declarados como tablas de BD ni archivos "
    "de entrada del proyecto. Sin ellos no se puede construir DATA_FM (SET data_fm_NAC data_fm_ext(DROP=VALOR_REL_VAL)) "
    "ni DATA_DEP_RUN."
)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #data_dep_run"))
sql_data_dep_run = """
SELECT t1.AÑO, t1.MES, t1.RUN_FONDO, t2.Sector, SUM(t1.VALOR_MERCADO) AS DATO
INTO #data_dep_run
FROM #data_fm t1
LEFT JOIN #id_fm t2 ON t1.RUN_FONDO = t2.RUN_FONDO
WHERE t1.T_INSTCORTO IN ('DPC', 'DPL') AND t1.MES IN (3, 6, 9, 12)
GROUP BY t1.AÑO, t1.MES, t1.RUN_FONDO, t2.Sector
"""
res = work_conn.execute(text(sql_data_dep_run))
_log("#data_dep_run", res.rowcount)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #data_dep"))
sql_data_dep = """
SELECT AÑO, MES / 3 AS TRIM, SECTOR, SUM(DATO) AS DATO
INTO #data_dep
FROM #data_dep_run
GROUP BY AÑO, MES, SECTOR
"""
res = work_conn.execute(text(sql_data_dep))
_log("#data_dep", res.rowcount)


In [ ]:
# 3. CALCULA DEPOSITOS A HOGARES
work_conn.execute(text("DROP TABLE IF EXISTS #dep_hh"))
sql_dep_hh = """
SELECT t1.AÑO, t1.MES / 3 AS TRIM, t2.SECTOR, SUM(t1.EST * t2.DATO) AS DATO
INTO #dep_hh
FROM #dato_est t1, #data_dep_run t2
WHERE t1.AÑO = t2.AÑO AND t1.MES = t2.MES AND t1.RUN_SDV = t2.RUN_FONDO AND t1.AÑO >= 2017
GROUP BY t1.AÑO, t1.MES, t2.SECTOR
"""
res = work_conn.execute(text(sql_dep_hh))
_log("#dep_hh", res.rowcount)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #dep_hh_2"))
sql_dep_hh_2 = """
SELECT t2.AÑO, t2.MES / 3 AS TRIM, t2.SECTOR, SUM(t1.EST * t2.DATO) AS DATO
INTO #dep_hh_2
FROM #dato_est t1, #data_dep_run t2
WHERE t1.AÑO = t2.AÑO AND t1.RUN_SDV = t2.RUN_FONDO AND t2.AÑO < 2017
GROUP BY t2.AÑO, t2.MES, t2.SECTOR
"""
res = work_conn.execute(text(sql_dep_hh_2))
res = work_conn.execute(text("INSERT INTO #dep_hh (AÑO, TRIM, SECTOR, DATO) SELECT AÑO, TRIM, SECTOR, DATO FROM #dep_hh_2"))
_log("INSERT #dep_hh desde #dep_hh_2", res.rowcount)


In [ ]:
# % total para dep de hogares
work_conn.execute(text("DROP TABLE IF EXISTS #dep_hh_fm"))
sql_dep_hh_fm = """
SELECT t1.AÑO, t1.TRIM, t1.Sector, t1.DATO / t2.DATO AS DATO
INTO #dep_hh_fm
FROM #dep_hh t1, #data_dep t2
WHERE t1.AÑO = t2.AÑO AND t1.TRIM = t2.TRIM AND t1.SECTOR = t2.SECTOR
"""
res = work_conn.execute(text(sql_dep_hh_fm))
_log("#dep_hh_fm", res.rowcount)


In [ ]:
# genera años faltantes: 2003, 2004 y 2005 replican el valor de 2006 T1
work_conn.execute(text("DROP TABLE IF EXISTS #imputa"))
sql_imputa = """
SELECT a.anio_imputado AS AÑO, t1.TRIM, t1.Sector, t1.DATO
INTO #imputa
FROM #dep_hh_fm t1
CROSS JOIN (SELECT 2005 AS anio_imputado UNION ALL SELECT 2004 UNION ALL SELECT 2003) a
WHERE t1.AÑO = 2006 AND t1.TRIM = 1
"""
res = work_conn.execute(text(sql_imputa))
_log("#imputa", res.rowcount)


In [ ]:
# imputa_a/b/c: el mismo bloque imputado con trimestre 2, 3 y 4
sql_imputa_trim = """
INSERT INTO #dep_hh_fm (AÑO, TRIM, Sector, DATO)
SELECT AÑO, :trim AS TRIM, Sector, DATO FROM #imputa
"""
res = work_conn.execute(text("INSERT INTO #dep_hh_fm (AÑO, TRIM, Sector, DATO) SELECT AÑO, TRIM, Sector, DATO FROM #imputa"))
for _trim in (2, 3, 4):
    res = work_conn.execute(text(sql_imputa_trim), {"trim": _trim})
# genera base serie completa
_log("#dep_hh_fm (serie completa)", res.rowcount)


In [ ]:
for t in ["#id_fm", "#dato_est", "#data_fm", "#data_dep_run", "#data_dep", "#dep_hh", "#dep_hh_2", "#imputa"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# para recalcular años de coyuntura o serie completa si es cierre de año
with engine.begin() as conn:
    res = conn.execute(text("DELETE FROM TABLAS.dbo.DEP_HH_FM WHERE AÑO >= :anio"), {"anio": anio})
    _log("DELETE TABLAS.dbo.DEP_HH_FM", res.rowcount)


In [ ]:
# para recalcular años de coyuntura o serie completa si es cierre de año
res = work_conn.execute(text("DELETE FROM #dep_hh_fm WHERE AÑO < :anio"), {"anio": anio})
_log("DELETE #dep_hh_fm (años previos)", res.rowcount)


In [ ]:
# APPEND FORCE: alinea por nombre de columna, server-side desde la #tmp
cols_dep_hh_fm = "AÑO, TRIM, Sector, DATO"
sql_append_dep_hh_fm = f"""
INSERT INTO TABLAS.dbo.DEP_HH_FM ({cols_dep_hh_fm})
SELECT {cols_dep_hh_fm}
FROM #dep_hh_fm
"""
res = work_conn.execute(text(sql_append_dep_hh_fm))
_log("APPEND TABLAS.dbo.DEP_HH_FM", res.rowcount)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #dep_hh_fm"))


## Bonos_Ext

Calcula los precios implícitos de bonos externos por año, trimestre y sector (balance final y balance de inicio) y reemplaza con ellos la tabla de precios de bonos externos

*confianza: medium · verificador: approve · SAS: PROC IMPORT (XLSX) + PROC SQL CREATE/UPDATE + DATA step de concatenación + carga a tabla de BD*

In [ ]:
# ========= Bonos_Ext =========
# DATA DE BONOS DE LA BALANZA DE PAGOS PARA CALCULAR PRECIOS IMPLÍCITOS A USAR EN LAS CUENTAS DE GOBIERNO, EMPRESAS Y HOLDINGS
# 1. IMPORTA DATA
ruta_bonos_ext = Path("data") / "INFO_AUX" / "bonos_ext_cdr18.xlsx"
bonos_ext = pd.read_excel(ruta_bonos_ext, sheet_name="DATA")
_log("bonos_ext", bonos_ext)


In [ ]:
# UPDATE BONOS_EXT SET CNSI=51021 WHERE CNSI=5102
bonos_ext.loc[bonos_ext["CNSI"] == 5102, "CNSI"] = 51021
# UPDATE BONOS_EXT SET CNSI=321 WHERE CNSI=322
bonos_ext.loc[bonos_ext["CNSI"] == 322, "CNSI"] = 321
# UPDATE BONOS_EXT SET C_CAGENTE='6' WHERE C_CAGENTE NOT IN ('6')
bonos_ext["C_CAGENTE"] = bonos_ext["C_CAGENTE"].astype(str)
bonos_ext.loc[bonos_ext["C_CAGENTE"] != "6", "C_CAGENTE"] = "6"
# incorporado cierre 2025q2
bonos_ext.loc[bonos_ext["CNSI"] == 33, "CNSI"] = 36
_log("bonos_ext", bonos_ext)


In [ ]:
bonos_ext_est = pd.read_excel(ruta_bonos_ext, sheet_name="DATA_EST")
# delete from BONOS_EXT_EST where año=. (elimina filas con año faltante)
bonos_ext_est = bonos_ext_est[bonos_ext_est["Año"].notna()].copy()
_log("bonos_ext_est", bonos_ext_est)


In [ ]:
# los datos importados de Excel viven en pandas: se suben a la sesión para operar en SQL
bonos_ext.to_sql("#bonos_ext", work_conn, if_exists="replace", index=False)
bonos_ext_est.to_sql("#bonos_ext_est", work_conn, if_exists="replace", index=False)


In [ ]:
# CALCULA PRECIOS PARA GOBIERNO, EMPRESAS Y HOLDINGS
# CIERRE 2021: INCORPORA TMB BANCOS
# CIERRE 2022Q2: INCORPORA PRECIO DE BNOS EMITIDOS EN EL EXTERIOR DE AUXILIARES (36)
work_conn.execute(text("DROP TABLE IF EXISTS #bonos_ext_precio"))
sql_bonos_ext_precio = """
SELECT [Año], [Trim], CNSI AS Sector, C_CAGENTE, C_SCN, C_CUENTA,
       SUM(Valor_Mercado) / NULLIF(SUM(Valor_par), 0) AS Precio
INTO #bonos_ext_precio
FROM #bonos_ext
WHERE fuente IN ('Mercado Externo', 'Mercado Externo (Recompras)')
  AND CNSI IN (41, 37, 5101, 51021, 5102, 321, 36)
GROUP BY [Año], [Trim], CNSI, C_CAGENTE, C_SCN, C_CUENTA
"""
res = work_conn.execute(text(sql_bonos_ext_precio))
_log("#bonos_ext_precio", res.rowcount)


In [ ]:
# Balance Final, para recompras en empresas
work_conn.execute(text("DROP TABLE IF EXISTS #bonos_ext_recompra"))
sql_bonos_ext_recompra = """
SELECT [Año], [Trim], CNSI AS Sector, '54' AS C_CAGENTE, C_SCN, C_CUENTA,
       SUM(Valor_Mercado) / NULLIF(SUM(Valor_par), 0) AS Precio
INTO #bonos_ext_recompra
FROM #bonos_ext
WHERE fuente IN ('Mercado Externo (Recompras)')
  AND CNSI IN (5101, 51021)
GROUP BY [Año], [Trim], CNSI, C_SCN, C_CUENTA
"""
res = work_conn.execute(text(sql_bonos_ext_recompra))
_log("#bonos_ext_recompra", res.rowcount)


In [ ]:
# update Bonos_Ext_Recompra set Precio=1 where Precio=.
res = work_conn.execute(text("UPDATE #bonos_ext_recompra SET Precio = 1 WHERE Precio IS NULL"))
_log("UPDATE #bonos_ext_recompra Precio=1", res.rowcount)


In [ ]:
# data Bonos_Ext_Precio; set Bonos_Ext_Recompra Bonos_Ext_Precio BONOS_EXT_EST;
cols_precio = "[Año], [Trim], Sector, C_CAGENTE, C_SCN, C_CUENTA, Precio"
work_conn.execute(text("DROP TABLE IF EXISTS #bonos_ext_precio_all"))
sql_concat_precio = f"""
SELECT {cols_precio} INTO #bonos_ext_precio_all FROM #bonos_ext_recompra
UNION ALL
SELECT {cols_precio} FROM #bonos_ext_precio
UNION ALL
SELECT {cols_precio} FROM #bonos_ext_est
"""
res = work_conn.execute(text(sql_concat_precio))
_log("#bonos_ext_precio_all", res.rowcount)


In [ ]:
# Balance Inicio: desplaza un trimestre (Trim=4 pasa al Trim 1 del año siguiente)
work_conn.execute(text("DROP TABLE IF EXISTS #bonos_ext_precio_2"))
sql_bonos_ext_precio_2 = """
SELECT (CASE WHEN [Trim] = 4 THEN [Año] + 1 ELSE [Año] END) AS [Año],
       (CASE WHEN [Trim] = 4 THEN 1 ELSE [Trim] + 1 END) AS [Trim],
       Sector, C_CAGENTE, C_SCN, 'Bce Inicio' AS C_CUENTA, Precio
INTO #bonos_ext_precio_2
FROM #bonos_ext_precio_all
"""
res = work_conn.execute(text(sql_bonos_ext_precio_2))
_log("#bonos_ext_precio_2", res.rowcount)


In [ ]:
# DATA tablas.Bonos_Ext_Precio; SET Bonos_Ext_Precio_2 Bonos_Ext_Precio;
# el SAS reescribía la tabla completa: DELETE sin WHERE + INSERT (conserva DDL y permisos)
res = work_conn.execute(text("DELETE FROM TABLAS.dbo.BONOS_EXT_PRECIO"))
_log("DELETE TABLAS.dbo.BONOS_EXT_PRECIO", res.rowcount)


In [ ]:
cols_destino = "[Año], [Trim], Sector, C_CAGENTE, C_SCN, C_CUENTA, Precio"
sql_insert_destino = f"""
INSERT INTO TABLAS.dbo.BONOS_EXT_PRECIO ({cols_destino})
SELECT {cols_destino} FROM #bonos_ext_precio_2
UNION ALL
SELECT {cols_destino} FROM #bonos_ext_precio_all
"""
res = work_conn.execute(text(sql_insert_destino))
_log("INSERT TABLAS.dbo.BONOS_EXT_PRECIO", res.rowcount)


In [ ]:
# update tablas.Bonos_Ext_Precio set Sector=36912 where Sector=36
with engine.begin() as conn:
    res = conn.execute(text("UPDATE TABLAS.dbo.BONOS_EXT_PRECIO SET Sector = 36912 WHERE Sector = 36"))
    _log("UPDATE TABLAS.dbo.BONOS_EXT_PRECIO Sector=36912", res.rowcount)


In [ ]:
# drop table Bonos_Ext_Precio, BONOS_EXT_EST, Bonos_Ext_Precio_2, bonos_ext, Bonos_Ext_Recompra
for t in ["#bonos_ext_precio", "#bonos_ext_precio_all", "#bonos_ext_est", "#bonos_ext_precio_2", "#bonos_ext", "#bonos_ext_recompra"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))
